# Research Paper RAG

Lookup information in research papers


In [1]:
import getpass
import os
from pathlib import Path
from typing import Any
from uuid import NAMESPACE_URL, uuid5

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_core.documents import Document
from langchain_core.globals import set_debug, set_verbose
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFaceEmbeddings,
    HuggingFaceEndpoint,
)
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone, ServerlessSpec
from pydantic import BaseModel, Field


set_debug(False)
set_verbose(False)

/var/folders/yf/87bpypv93ks0mnh8xhx5p0yw0000gn/T/ipykernel_38377/3204453601.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
/Users/yin/Documents/projects/research-paper-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Setup Pinecone for Vector database and Huggingface for open sourced models

In [2]:
# load environment variables from .env file
load_dotenv()

True

In [3]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "research-paper-rag-bge-m3")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "papers-v1")

EMBEDDING_MODEL = os.getenv("HF_EMBEDDING_MODEL", "BAAI/bge-m3")
EMBEDDING_DIMENSION = 1024  # output dimension

CHAT_MODEL_ID = "deepseek-ai/DeepSeek-V4-Pro"
HF_PROVIDER = "auto"
RERANK_MODEL = "bge-reranker-v2-m3"  # best for short queries and long replies

RETRIEVAL_TOP_K = 20
RERANK_TOP_N = 5

print(f"Data directory: {DATA_DIR}")
print(f"Pinecone index: {INDEX_NAME} / namespace: {NAMESPACE}")
print(f"Embedding model: {EMBEDDING_MODEL} ({EMBEDDING_DIMENSION} dimensions)")
print(f"Chat model: {CHAT_MODEL_ID} via provider={HF_PROVIDER}")

Data directory: /Users/yin/Documents/projects/research-paper-rag/data
Pinecone index: research-paper-rag-bge-m3 / namespace: papers-v1
Embedding model: BAAI/bge-m3 (1024 dimensions)
Chat model: deepseek-ai/DeepSeek-V4-Pro via provider=auto


## 2. Load and Chunk PDFs

In [4]:
# load pdfs

loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True,
)
raw_docs = loader.load()

print(f"Loaded {len(raw_docs)} pages from {DATA_DIR}")
print(raw_docs[0].metadata if raw_docs else "No documents found")

100%|██████████| 5/5 [00:00<00:00, 11.13it/s]

Loaded 109 pages from /Users/yin/Documents/projects/research-paper-rag/data
{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '', 'source': '/Users/yin/Documents/projects/research-paper-rag/data/GLINER2.pdf', 'file_path': '/Users/yin/Documents/projects/research-paper-rag/data/GLINER2.pdf', 'total_pages': 11, 'format': 'PDF 1.5', 'title': 'GLiNER2: An Efficient Multi-Task Information Extraction System with Schema-Driven Interface', 'author': 'Urchade Zaratiana; Gil Pasternak; Oliver Boyd; George Hurn-Maloney; Ash Lewis', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}


In [5]:
# split pages into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

splits = text_splitter.split_documents(raw_docs)

for chunk_number, doc in enumerate(splits):
    source_path = Path(str(doc.metadata.get("source", "unknown")))
    page = doc.metadata.get("page", None)
    doc.metadata.update(
        {
            "source": source_path.name,
            "source_path": str(source_path),
            "page": int(page) + 1 if isinstance(page, int) else page,
            "chunk_number": chunk_number,
            "namespace": NAMESPACE,
        }
    )

print(f"Created {len(splits)} chunks")
print(splits[0].page_content[:500] if splits else "No chunks created")
print(splits[0].metadata if splits else {})

Created 620 chunks
GLiNER2: An Efficient Multi-Task Information Extraction System
with Schema-Driven Interface
Urchade Zaratiana, Gil Pasternak, Oliver Boyd
George Hurn-Maloney, Ash Lewis
Fastino AI
{uz,gil,o8,g,ash}@fastino.ai
Abstract
Information extraction (IE) is fundamental to
numerous NLP applications, yet existing solu-
tions often require specialized models for differ-
ent tasks or rely on computationally expensive
large language models. We present GLiNER2,
a unified framework that enhances the orig-
inal 
{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '', 'source': 'GLINER2.pdf', 'file_path': '/Users/yin/Documents/projects/research-paper-rag/data/GLINER2.pdf', 'total_pages': 11, 'format': 'PDF 1.5', 'title': 'GLiNER2: An Efficient Multi-Task Information Extraction System with Schema-Driven Interface', 'author': 'Urchade Zaratiana; Gil Pasternak; Oliver Boyd; George Hurn-Maloney; Ash Lewis', 'subject': '', 'keywords': '', 'moddate': '', 'tr

In [6]:
# create unique ID for each chunk based on PDF source, page, chunk number, and the first 200 characters of content.
def stable_chunk_id(doc: Document) -> str:
    source = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", "unknown")
    chunk_number = doc.metadata.get("chunk_number", "unknown")
    content_fingerprint = doc.page_content[:200]
    return str(
        uuid5(NAMESPACE_URL, f"{source}:{page}:{chunk_number}:{content_fingerprint}")
    )


chunk_ids = [stable_chunk_id(doc) for doc in splits]
print("example chunk_ids:", chunk_ids[:3])

example chunk_ids: ['c78632b4-598b-5a5a-9497-d1efac76f78d', 'bbc79c3c-ab47-5099-8d10-60ced340c049', 'dd515802-1b68-54bb-b24c-1fcbd4800876']


## 3. Configure Embeddings and Pinecone

In [7]:
# define the embeddings model
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32,
    },
)

# create Pinecone client
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        vector_type="dense",
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        deletion_protection="disabled",
        tags={"project": "research-paper-rag", "embedding_model": EMBEDDING_MODEL},
    )

index = pc.Index(INDEX_NAME)
# use LangChain’s PineconeVectorStore wrapper
vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE,
)

index.describe_index_stats()

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 98414.12it/s]


{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'papers-v1': {'vector_count': 497}},
 'total_vector_count': 497,
 'vector_type': 'dense'}

In [8]:
if splits:
    upserted_ids = vector_store.add_documents(documents=splits, ids=chunk_ids)
    print(f"Upserted {len(upserted_ids)} chunks into namespace '{NAMESPACE}'")
else:
    print("No chunks to upsert")

index.describe_index_stats()

Upserted 620 chunks into namespace 'papers-v1'


{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'papers-v1': {'vector_count': 1117}},
 'total_vector_count': 1117,
 'vector_type': 'dense'}

## 4. Retrieve and Rerank

Two stage retrieval.

In [9]:
def format_source(doc: Document) -> str:
    source = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", "unknown")
    chunk = doc.metadata.get("chunk_number", "unknown")
    score = doc.metadata.get("rerank_score")
    score_text = f", rerank={score:.4f}" if isinstance(score, float) else ""
    return f"{source}, page {page}, chunk {chunk}{score_text}"


def rerank_documents(
    query: str, docs: list[Document], top_n: int = RERANK_TOP_N
) -> list[Document]:
    if not docs:
        return []

    documents = [
        {
            "id": str(i),
            "chunk_text": doc.page_content,
            "source": doc.metadata.get("source", "unknown"),
            "page": doc.metadata.get("page", "unknown"),
        }
        for i, doc in enumerate(docs)
    ]

    try:
        result = pc.inference.rerank(
            model=RERANK_MODEL,
            query=query,
            documents=documents,
            top_n=min(top_n, len(documents)),
            rank_fields=["chunk_text"],
            return_documents=True,
            parameters={"truncate": "END"},
        )
    except Exception as exc:
        print(f"Reranking skipped: {exc}")
        return docs[:top_n]

    reranked = []
    for item in result.data:
        original = docs[item.index]
        original.metadata["rerank_score"] = float(item.score)
        reranked.append(original)
    return reranked


def retrieve(query: str, filters: dict[str, Any] | None = None) -> list[Document]:
    candidates = vector_store.similarity_search(
        query,
        k=RETRIEVAL_TOP_K,
        filter=filters,
        namespace=NAMESPACE,
    )
    return rerank_documents(query, candidates, top_n=RERANK_TOP_N)

In [10]:
query = "What are the categories of attentional models?"
# retrieve relevant candidates from pinecone and rerank them
retrieved_docs = retrieve(query)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n[{i}] {format_source(doc)}")
    print(doc.page_content[:700])


[1] Saliency Prediction in Deep Learning Era.pdf, page 1.0, chunk 244.0, rerank=0.9985
cognitive resources to the most pertinent subsets of sensory
data. It acts as a shiftable information processing bottleneck,
allowing only objects within a circumscribed region to reach
higher levels of processing and visual awareness [1].
Broadly speaking, the literature on attentional models
can be split into two categories: task-agnostic approaches
(i.e. ﬁnding the salient pieces of information, a.k.a bottom-up
(BU) saliency [1]–[4]) and task-speciﬁc methods (i.e. ﬁnding
information relevant to the ongoing behavior, task, or goal
[5], [6]). Bottom-up salience is the most extensively studied
aspect of visual guidance. The model by Itti et al. [3] in 1998
triggered a lot of interests in vis

[2] Saliency Prediction in Deep Learning Era.pdf, page 1.0, chunk 184.0, rerank=0.9985
cognitive resources to the most pertinent subsets of sensory
data. It acts as a shiftable information processing bottleneck

## 5. Generate a Cited Answer

The model is instructed to answer only from retrieved context and return a validated Pydantic schema.

In [11]:
class SourceCitation(BaseModel):
    source: str = Field(description="PDF file name")
    page: int | str = Field(description="Page number if available")
    chunk_number: int | str = Field(description="Chunk number if available")
    quote: str = Field(description="Short supporting quote from the retrieved chunk")


class RagAnswer(BaseModel):
    question: str = Field(description="Original user question")
    answer: str = Field(description="Grounded answer based only on retrieved context")
    sources: list[SourceCitation] = Field(description="Sources used for the answer")
    confidence: str = Field(description="One of: high, medium, low")


parser = PydanticOutputParser(pydantic_object=RagAnswer)
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"SourceCitation": {"properties": {"source": {"description": "PDF file name", "title": "Source", "type": "string"}, "page": {"anyOf": [{"type": "integer"}, {"type": "string"}], "description": "Page number if available", "title": "Page"}, "chunk_number": {"anyOf": [{"type": "integer"}, {"type": "string"}], "description": "Chunk number if available", "title": "Chunk Number"}, "quote": {"description": "Short supporting quote from the retrieved chunk", "title": "Quote", "type": "string"}}, "required": ["source", "page", "chunk_number", "

In [12]:
llm = HuggingFaceEndpoint(
    repo_id=CHAT_MODEL_ID,
    task="text-generation",
    provider=HF_PROVIDER,
    max_new_tokens=700,
    do_sample=False,
    repetition_penalty=1.03,
)
chat_model = ChatHuggingFace(llm=llm)

In [13]:
def format_context(docs: list[Document]) -> str:
    blocks = []
    for i, doc in enumerate(docs, start=1):
        blocks.append(f"[Source {i}] {format_source(doc)}\n{doc.page_content}")
    return "\n\n".join(blocks)


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You answer questions about research papers. Use only the provided context. "
            "If the context is insufficient, say you are unsure. Include concise citations.\n\n"
            "{format_instructions}",
        ),
        (
            "human",
            "Question:\n{question}\n\nRetrieved context:\n{context}",
        ),
    ]
)

rag_chain = prompt | chat_model | parser

In [14]:
question = "What are the categories of attentional models?"
context_docs = retrieve(question)

answer = rag_chain.invoke(
    {
        "question": question,
        "context": format_context(context_docs),
        "format_instructions": parser.get_format_instructions(),
    }
)

answer

RagAnswer(question='What are the categories of attentional models?', answer='The literature on attentional models can be split into two broad categories: task-agnostic approaches (i.e., finding salient pieces of information, also known as bottom-up saliency) and task-specific methods (i.e., finding information relevant to ongoing behavior, task, or goal). Additionally, classic bottom-up saliency models themselves fall into different categories such as Bayesian, learning-based, spectral, and cognitive.', sources=[SourceCitation(source='Saliency Prediction in Deep Learning Era.pdf', page=1, chunk_number=244, quote='Broadly speaking, the literature on attentional models can be split into two categories: task-agnostic approaches (i.e. finding the salient pieces of information, a.k.a bottom-up (BU) saliency [1]–[4]) and task-specific methods (i.e. finding information relevant to the ongoing behavior, task, or goal [5], [6]).'), SourceCitation(source='Saliency Prediction in Deep Learning Era

In [15]:
print(answer.answer)
print("\nSources:")
for source in answer.sources:
    print(
        f"- {source.source}, page {source.page}, chunk {source.chunk_number}: {source.quote}"
    )

The literature on attentional models can be split into two broad categories: task-agnostic approaches (i.e., finding salient pieces of information, also known as bottom-up saliency) and task-specific methods (i.e., finding information relevant to ongoing behavior, task, or goal). Additionally, classic bottom-up saliency models themselves fall into different categories such as Bayesian, learning-based, spectral, and cognitive.

Sources:
- Saliency Prediction in Deep Learning Era.pdf, page 1, chunk 244: Broadly speaking, the literature on attentional models can be split into two categories: task-agnostic approaches (i.e. finding the salient pieces of information, a.k.a bottom-up (BU) saliency [1]–[4]) and task-specific methods (i.e. finding information relevant to the ongoing behavior, task, or goal [5], [6]).
- Saliency Prediction in Deep Learning Era.pdf, page 3, chunk 204: Rooted in these works, several saliency models have been proposed. They fall into different categories (e.g. Baye

## 6. Compare Answers With and Without RAG

Use the same question for a baseline model answer and a grounded RAG answer. This makes it easier to see what retrieval changes: answer specificity, uncertainty, and citations.


In [16]:
from textwrap import dedent

from IPython.display import Markdown, display


comparison_question = "What are the categories of attentional models?"

baseline_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question from your general model knowledge only. "
            "Do not use retrieved context or fabricate citations. If you are unsure, say so.",
        ),
        ("human", "Question:\n{question}"),
    ]
)
baseline_chain = baseline_prompt | chat_model


def response_text(response: Any) -> str:
    return str(getattr(response, "content", response)).strip()


def citation_lines(rag_answer: RagAnswer) -> str:
    if not rag_answer.sources:
        return "- No citations returned"
    return "\n".join(
        f"- {source.source}, page {source.page}, chunk {source.chunk_number}: {source.quote}"
        for source in rag_answer.sources
    )


baseline_answer = response_text(
    baseline_chain.invoke({"question": comparison_question})
)

comparison_docs = retrieve(comparison_question)
rag_comparison_answer = rag_chain.invoke(
    {
        "question": comparison_question,
        "context": format_context(comparison_docs),
        "format_instructions": parser.get_format_instructions(),
    }
)

display(
    Markdown(
        dedent(
            f"""
            ### Question
            {comparison_question}

            ### Before RAG: model-only answer
            {baseline_answer}

            ### After RAG: retrieved and cited answer
            {rag_comparison_answer.answer}

            **Confidence:** {rag_comparison_answer.confidence}

            **Sources**
            {citation_lines(rag_comparison_answer)}
            """
        ).strip()
    )
)


### Question
            What are the categories of attentional models?

            ### Before RAG: model-only answer
            Attention is a fundamental cognitive process that can be categorized in several ways within psychology and cognitive science. The main categories of attentional models are typically organized by function, mechanism, or type of attention. Here are the primary categories:

1. **Selective Attention Models**
   - **Early Selection Models** (e.g., Broadbent’s Filter Model) – suggest that irrelevant information is filtered out early, before semantic processing.
   - **Late Selection Models** (e.g., Deutsch & Deutsch) – propose that all inputs are processed for meaning, but only relevant information reaches awareness or response.
   - **Attenuation Model** (Treisman) – a compromise where unattended information is attenuated rather than completely blocked.

2. **Capacity / Resource Models**
   - **Limited-Capacity Models** – attention is a limited resource that can be allocated flexibly (e.g., Kahneman’s Capacity Model).
   - **Multiple Resource Theory** (Wickens) – proposes separate pools of attentional resources for different modalities or processing stages.

3. **Feature Integration Theory** (Treisman & Gelade)
   - Distinguishes between pre-attentive processing (automatic detection of basic features) and focused attention (binding features into objects).

4. **Sustained Attention (Vigilance) Models**
   - Concerned with the ability to maintain focus over prolonged periods, often studied in relation to signal detection theory.

5. **Divided Attention Models**
   - Examine how attention is distributed across multiple tasks, often related to automaticity (e.g., Shiffrin & Schneider’s dual-process theory).

6. **Executive Attention / Supervisory Attentional System**
   - Part of Baddeley’s working memory model and Norman & Shallice’s model, involving higher-order control, planning, and inhibition.

7. **Clinical/Pathological Attention Models**
   - Address attentional deficits in conditions like ADHD, neglect syndrome, or schizophrenia.

These categories are not mutually exclusive and often overlap in explaining how attention operates in different contexts.

            ### After RAG: retrieved and cited answer
            The literature broadly splits attentional models into two categories: task-agnostic approaches (bottom-up saliency) and task-specific methods. Additionally, bottom-up saliency models have been categorized into groups such as Bayesian, learning-based, spectral, and cognitive.

            **Confidence:** high

            **Sources**
            - Saliency Prediction in Deep Learning Era.pdf, page 1, chunk 244: Broadly speaking, the literature on attentional models can be split into two categories: task-agnostic approaches (i.e. ﬁnding the salient pieces of information, a.k.a bottom-up (BU) saliency [1]–[4]) and task-speciﬁc methods (i.e. ﬁnding information relevant to the ongoing behavior, task, or goal [5], [6]).
- Saliency Prediction in Deep Learning Era.pdf, page 3, chunk 264: Rooted in these works, several saliency models have been proposed. They fall into different categories (e.g. Bayesian, learning-based, spectral, cognitive), according to our study

## 7. Quick Retrieval Evaluation

Use a small, stable question set. Start by checking whether the right document/page appears after retrieval and reranking, then expand to answer-level evaluation.

In [17]:
eval_questions = [
    {
        "question": "What are the categories of attentional models?",
        "expected_source_contains": "",
        "expected_terms": ["attentional", "model"],
    },
    {
        "question": "Explain saliency detection.",
        "expected_source_contains": "",
        "expected_terms": ["saliency", "detection"],
    },
]


def evaluate_retrieval_case(case: dict[str, Any]) -> dict[str, Any]:
    docs = retrieve(case["question"])
    joined = "\n".join(doc.page_content.lower() for doc in docs)
    source_names = [str(doc.metadata.get("source", "")) for doc in docs]
    expected_source = case.get("expected_source_contains", "").lower()
    expected_terms = [term.lower() for term in case.get("expected_terms", [])]
    return {
        "question": case["question"],
        "retrieved": len(docs),
        "top_source": source_names[0] if source_names else None,
        "source_match": (not expected_source)
        or any(expected_source in src.lower() for src in source_names),
        "term_recall": sum(term in joined for term in expected_terms),
        "expected_terms": len(expected_terms),
    }


retrieval_eval = [evaluate_retrieval_case(case) for case in eval_questions]
retrieval_eval

[{'question': 'What are the categories of attentional models?',
  'retrieved': 5,
  'top_source': 'Saliency Prediction in Deep Learning Era.pdf',
  'source_match': True,
  'term_recall': 2,
  'expected_terms': 2},
 {'question': 'Explain saliency detection.',
  'retrieved': 5,
  'top_source': 'Salient object detection A benchmark.pdf',
  'source_match': True,
  'term_recall': 2,
  'expected_terms': 2}]